# Chakritha – Classical Agent Implementation

## Project Objective
Implement five classical AI agents for production incident analysis.

## Responsibilities
1. Simple Reflex Agent
2. Model-Based Reflex Agent
3. Goal-Based Agent
4. Utility-Based Agent
5. Learning Agent with feedback mechanism

All agents will use the shared incident scenarios and the team's common AgentResult interface.

In [1]:
# STEP 7.2: CONNECT TO SHARED PROJECT

import sys
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\CHAKRITHA\Documents\Python Files- BITS\BITS material\Agentic AI\Assignment1"
)

assert PROJECT_ROOT.exists(), "Project folder not found!"
assert (PROJECT_ROOT / "incd_assist" / "contracts.py").exists(), "Shared contracts not found!"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import shared project components
from incd_assist.scenarios import load_scenarios
from incd_assist.contracts import IncidentScenario, AgentResult, Agent

from incd_assist.tools import (
    get_metrics,
    search_logs,
    get_deployment_history,
    check_previous_version
)

# Load scenarios A, B and C
scenarios = load_scenarios()

print("=" * 60)
print("SHARED PROJECT CONNECTION")
print("=" * 60)

print("Python:", sys.version.split()[0])
print("Project:", PROJECT_ROOT)
print("Scenario IDs:", list(scenarios.keys()))

for scenario_id, scenario in scenarios.items():
    print("\nScenario:", scenario_id)
    print("Name:", scenario.label)
    print("Metrics:", dict(scenario.metrics))
    print("Number of logs:", len(scenario.logs))
    print("Deployment records:", len(scenario.deployment_history))

print("\nSUCCESS: Shared project components imported!")
print("SUCCESS: All scenarios loaded!")

SHARED PROJECT CONNECTION
Python: 3.11.9
Project: C:\Users\CHAKRITHA\Documents\Python Files- BITS\BITS material\Agentic AI\Assignment1
Scenario IDs: ['A', 'B', 'C']

Scenario: A
Name: API latency and DB pool exhaustion
Metrics: {'latency_p95_ms': 1400, 'error_rate_pct': 18.5, 'cpu_pct': 62, 'requests_per_min': 1800, 'db_connections_in_use': 100, 'db_pool_limit': 100, 'availability_pct': 81.5}
Number of logs: 3
Deployment records: 1

Scenario: B
Name: High traffic and CPU with healthy DB pool
Metrics: {'latency_p95_ms': 480, 'error_rate_pct': 0.3, 'cpu_pct': 92, 'requests_per_min': 5200, 'db_connections_in_use': 30, 'db_pool_limit': 100, 'availability_pct': 99.9}
Number of logs: 3
Deployment records: 1

Scenario: C
Name: Startup failure after configuration change
Metrics: {'latency_p95_ms': 0, 'error_rate_pct': 100.0, 'cpu_pct': 7, 'requests_per_min': 0, 'db_connections_in_use': 2, 'db_pool_limit': 100, 'availability_pct': 0.0}
Number of logs: 3
Deployment records: 2

SUCCESS: Shared pr

## 1. Simple Reflex Agent

The Simple Reflex Agent makes decisions using current
metrics and log evidence.

It follows predefined IF-THEN rules.

It does not use memory, past experience, or an LLM.

In [26]:
# STEP 7.3: SIMPLE REFLEX AGENT

class SimpleReflexAgent:

    def decide(self, scenario: IncidentScenario) -> AgentResult:

        metrics = scenario.metrics
        logs = scenario.logs

        latency = metrics.get("latency_p95_ms", 0)
        error_rate = metrics.get("error_rate_pct", 0)
        cpu = metrics.get("cpu_pct", 0)
        requests = metrics.get("requests_per_min", 0)
        db_connections = metrics.get("db_connections_in_use", 0)
        db_limit = metrics.get("db_pool_limit", 0)
        availability = metrics.get("availability_pct", 0)

        log_text = " ".join(logs).lower()

        # RULE 1: Database connection pool exhaustion
        if (
            db_limit > 0
            and db_connections >= db_limit
            and (latency >= 1000 or error_rate >= 5)
        ):

            cause = "Database connection pool exhaustion"

            summary = (
                "API requests are experiencing delays because "
                "the database connection pool is fully occupied."
            )

            action = (
                "Investigate long-running database connections, "
                "connection wait queues and pool capacity. "
                "Review safe capacity adjustments."
            )

            evidence = (
                f"Database connections: {db_connections}/{db_limit}",
                f"P95 latency: {latency} ms",
                f"Error rate: {error_rate}%",
                "Database pool exhaustion detected by predefined rule."
            )

            reason = "Rule 1 matched: Database pool is full."

                # RULE 2: Application startup/configuration failure
        elif (
            "startup failed" in log_text
            or "missing auth_issuer" in log_text
        ):

            deployments = [
                dict(item) for item in scenario.deployment_history
            ]

            previous = dict(scenario.previous_version)

            failed = next(
                (
                    item for item in reversed(deployments)
                    if "failed" in str(item.get("status", "")).lower()
                ),
                {}
            )

            failed_version = failed.get("version", "unknown")
            failed_change = failed.get("change", "unknown")
            previous_version = previous.get("version", "unknown")

            cause = "Application startup failure due to AUTH_ISSUER configuration change"

            summary = (
                f"Application deployment {failed_version} failed to start "
                "because the AUTH_ISSUER configuration is missing."
            )

            action = (
                f"Compare the failed configuration with the previous "
                f"healthy version {previous_version}. After operator "
                f"approval, restore the correct configuration or perform "
                f"a controlled rollback. Verify application startup "
                f"and health metrics."
            )

            evidence = (
                f"Failed deployment: {failed_version}",
                f"Configuration change: {failed_change}",
                f"Previous healthy version: {previous_version}",
                f"Previous version health: {previous.get('health')}",
                f"Previous configuration: {previous.get('config')}",
                "Startup failure found in application logs."
                "The earlier DB transient timeout was marked resolved; "
                "it is historical and inconclusive for the current startup failure."
            )

            reason = (
                "Rule 2 matched: Missing AUTH_ISSUER configuration "
                "after deployment."
            )
            
        # RULE 3: High traffic and CPU utilization
        elif (
            cpu >= 85
            and requests >= 4000
            and db_limit > 0
            and db_connections < db_limit * 0.8
        ):

            cause = "High traffic causing CPU saturation"

            summary = (
                "Incoming traffic is increasing CPU utilization "
                "while database connections remain healthy."
            )

            action = (
                "Review CPU utilization, request handling and "
                "horizontal scaling capacity."
            )

            evidence = (
                f"CPU utilization: {cpu}%",
                f"Requests per minute: {requests}",
                f"Database connections: {db_connections}/{db_limit}",
                "Database pool is not exhausted."
            )

            reason = "Rule 3 matched: High traffic and CPU utilization."

        # RULE 4: Unknown incident
        else:

            cause = "Unknown or insufficiently classified incident"

            summary = "Available evidence does not match known rules."

            action = (
                "Collect additional monitoring metrics and log evidence "
                "before recommending corrective action."
            )

            evidence = (
                f"P95 latency: {latency} ms",
                f"CPU utilization: {cpu}%",
                f"Error rate: {error_rate}%"
            )

            reason = "No predefined rule matched."

        return AgentResult(
            agent_name="Simple Reflex Agent",
            incident_summary=summary,
            evidence=evidence,
            likely_cause=cause,
            recommended_action=action,
            verification_steps=(
                "Review the relevant monitoring metrics.",
                "Inspect the corresponding application logs.",
                "Confirm recovery using availability and error-rate metrics."
            ),
            limitations=(
                "Uses predefined IF-THEN rules only.",
                "Does not learn from previous incidents.",
                "Does not maintain an internal memory or historical state."
            ),
            reason=reason
        )


print("Simple Reflex Agent class created successfully!")

Simple Reflex Agent class created successfully!


In [8]:
# TEST SIMPLE REFLEX AGENT ON SCENARIOS A, B AND C

simple_reflex_agent = SimpleReflexAgent()

simple_reflex_results = {}

for scenario_id, scenario in scenarios.items():

    result = simple_reflex_agent.decide(scenario)

    simple_reflex_results[scenario_id] = result

    print("\n" + "=" * 65)
    print("SCENARIO:", scenario_id)
    print("=" * 65)

    print("Agent:", result.agent_name)
    print("Summary:", result.incident_summary)
    print("Likely Cause:", result.likely_cause)
    print("Recommended Action:", result.recommended_action)
    print("Reason:", result.reason)

    print("\nEvidence:")
    for item in result.evidence:
        print("-", item)

print("\n" + "=" * 65)
print("SIMPLE REFLEX AGENT TESTING COMPLETED")
print("Total scenarios tested:", len(simple_reflex_results))
print("=" * 65)


SCENARIO: A
Agent: Simple Reflex Agent
Summary: API requests are experiencing delays because the database connection pool is fully occupied.
Likely Cause: Database connection pool exhaustion
Recommended Action: Investigate long-running database connections, connection wait queues and pool capacity. Review safe capacity adjustments.
Reason: Rule 1 matched: Database pool is full.

Evidence:
- Database connections: 100/100
- P95 latency: 1400 ms
- Error rate: 18.5%
- Database pool exhaustion detected by predefined rule.

SCENARIO: B
Agent: Simple Reflex Agent
Summary: Incoming traffic is increasing CPU utilization while database connections remain healthy.
Likely Cause: High traffic causing CPU saturation
Recommended Action: Review CPU utilization, request handling and horizontal scaling capacity.
Reason: Rule 3 matched: High traffic and CPU utilization.

Evidence:
- CPU utilization: 92%
- Requests per minute: 5200
- Database connections: 30/100
- Database pool is not exhausted.

SCENARI

In [9]:
# INSPECT SCENARIO C RECOVERY EVIDENCE

scenario_c = scenarios["C"]

print("SCENARIO C LOGS")
for log in scenario_c.logs:
    print("-", log)

print("\nDEPLOYMENT HISTORY")
for deployment in scenario_c.deployment_history:
    print(dict(deployment))

print("\nPREVIOUS HEALTHY VERSION")
print(dict(scenario_c.previous_version))

SCENARIO C LOGS
- 2026-11-16T09:10:00Z WARN db transient timeout on previous release (resolved)
- 2026-11-16T10:00:00Z ERROR app startup failed: missing AUTH_ISSUER setting
- 2026-11-16T10:00:03Z WARN health check unavailable; no live request logs

DEPLOYMENT HISTORY
{'version': '2.4.1', 'status': 'failed startup', 'change': 'AUTH_ISSUER configuration changed at 10:00Z'}
{'version': '2.4.0', 'status': 'healthy', 'change': 'previous running version'}

PREVIOUS HEALTHY VERSION
{'version': '2.4.0', 'health': 'healthy before config change', 'config': 'AUTH_ISSUER present'}


## 2. Model-Based Reflex Agent

The Model-Based Reflex Agent maintains an internal state
of each incident.

It remembers previous observations, tracks changes in
metrics, and uses deployment and log history to improve
its incident analysis.

In [27]:
# STEP 7.4: MODEL-BASED REFLEX AGENT

class ModelBasedReflexAgent:

    def __init__(self):
        self.internal_state = {}

    def decide(self, scenario: IncidentScenario) -> AgentResult:

        sid = scenario.id
        metrics = scenario.metrics
        logs = list(scenario.logs)
        log_text = " ".join(logs).lower()

        # Retrieve previous internal state
        previous = self.internal_state.get(sid, {})
        observation_count = previous.get("observation_count", 0) + 1

        latency = metrics.get("latency_p95_ms", 0)
        error_rate = metrics.get("error_rate_pct", 0)
        cpu = metrics.get("cpu_pct", 0)
        requests = metrics.get("requests_per_min", 0)
        db_connections = metrics.get("db_connections_in_use", 0)
        db_limit = metrics.get("db_pool_limit", 0)

        evidence = []

        # RULE 1: Startup configuration failure
        if "startup failed" in log_text or "missing auth_issuer" in log_text:

            deployments = [
                dict(item) for item in scenario.deployment_history
            ]

            previous_version = dict(scenario.previous_version)

            failed = next(
                (
                    item for item in reversed(deployments)
                    if "failed" in item.get("status", "").lower()
                ),
                {}
            )

            failed_version = failed.get("version", "unknown")
            healthy_version = previous_version.get("version", "unknown")

            cause = "Startup failure caused by missing AUTH_ISSUER configuration"

            summary = (
                f"Deployment {failed_version} failed during startup "
                "following a configuration change."
            )

            action = (
                f"Verify AUTH_ISSUER and compare the configuration with "
                f"the previous healthy version {healthy_version}. "
                "After operator approval, restore the last-known-good "
                "configuration if the failure is confirmed."
            )

            evidence.extend([
                f"Failed deployment: {failed_version}",
                f"Previous healthy version: {healthy_version}",
                f"Previous configuration: {previous_version.get('config')}"
            ])

            # Maintain chronological incident context
            stale_db_warning = any(
                "db transient timeout" in log.lower()
                and "resolved" in log.lower()
                for log in logs
            )

            if stale_db_warning:
                evidence.append(
                    "The earlier DB transient timeout was marked resolved; "
                    "it is historical and inconclusive for the current startup failure."
                )

            reason = (
                "Internal model identified a configuration failure "
                "after comparing the deployment history and log timeline."
            )

        # RULE 2: Database connection pool exhaustion
        elif (
            db_limit > 0
            and db_connections >= db_limit
            and (latency >= 1000 or error_rate >= 5)
        ):

            cause = "Database connection pool exhaustion"

            summary = (
                "The database connection pool is fully occupied, "
                "causing API delays and errors."
            )

            action = (
                "Investigate long-running connections and checkout queues. "
                "Review safe pool capacity adjustments."
            )

            evidence.extend([
                f"Database connections: {db_connections}/{db_limit}",
                f"P95 latency: {latency} ms",
                f"Error rate: {error_rate}%"
            ])

            reason = "Internal model identified database pool saturation."

        # RULE 3: High traffic and CPU
        elif (
            cpu >= 85
            and requests >= 4000
            and db_limit > 0
            and db_connections < db_limit * 0.8
        ):

            cause = "Traffic-driven CPU saturation, not DB exhaustion"

            summary = (
                "High incoming traffic is increasing CPU utilization "
                "while the database pool remains healthy."
            )

            action = (
                "Monitor application CPU and consider controlled "
                "horizontal scaling."
            )

            evidence.extend([
                f"CPU utilization: {cpu}%",
                f"Requests per minute: {requests}",
                f"Database connections: {db_connections}/{db_limit}"
            ])

            reason = "Internal model identified high traffic and CPU pressure."

        else:

            cause = "Unknown or insufficiently classified incident"

            summary = "Available evidence does not match known incident rules."

            action = "Collect additional monitoring and log evidence."

            evidence.append("No known incident pattern was identified.")

            reason = "Insufficient evidence for a confident classification."

        # Compare current metrics with previous internal observations
        if previous:

            changes = [
                f"{key}: {previous['metrics'].get(key)} -> {value}"
                for key, value in metrics.items()
                if previous["metrics"].get(key) != value
            ]

            evidence.append(
                "Previous observation: incident state was "
                + previous["incident_state"]
            )

            evidence.append(
                "Metric changes: " + (
                    "; ".join(changes) if changes else "No metric changes detected."
                )
            )

        else:
            evidence.append(
                "First observation: initial internal state created."
            )

        evidence.append(
            f"Internal observation number: {observation_count}"
        )

        # Update internal memory
        self.internal_state[sid] = {
            "observation_count": observation_count,
            "incident_state": cause,
            "metrics": dict(metrics),
            "likely_cause": cause
        }

        return AgentResult(
            agent_name="Model-Based Reflex Agent",
            incident_summary=summary,
            evidence=tuple(evidence),
            likely_cause=cause,
            recommended_action=action,
            verification_steps=(
                "Compare current and previous monitoring observations.",
                "Inspect relevant application logs and deployment history.",
                "Verify service recovery using health and availability metrics."
            ),
            limitations=(
                "Uses a simple internal state rather than a learned model.",
                "Depends on the quality of available logs and metrics.",
                "Does not automatically execute recovery actions."
            ),
            reason=reason
        )


print("Model-Based Reflex Agent class created successfully!")

Model-Based Reflex Agent class created successfully!


In [12]:
# TEST MODEL-BASED REFLEX AGENT

model_based_agent = ModelBasedReflexAgent()

model_based_results = {}

for scenario_id, scenario in scenarios.items():

    result = model_based_agent.decide(scenario)

    model_based_results[scenario_id] = result

    print("\n" + "=" * 65)
    print("SCENARIO:", scenario_id)
    print("=" * 65)

    print("Agent:", result.agent_name)
    print("Summary:", result.incident_summary)
    print("Likely Cause:", result.likely_cause)
    print("Recommended Action:", result.recommended_action)
    print("Reason:", result.reason)

    print("\nEvidence:")
    for item in result.evidence:
        print("-", item)

print("\n" + "=" * 65)
print("MODEL-BASED AGENT TESTING COMPLETED")
print("Total scenarios tested:", len(model_based_results))
print("Internal memory records:", len(model_based_agent.internal_state))
print("=" * 65)


SCENARIO: A
Agent: Model-Based Reflex Agent
Summary: The database connection pool is fully occupied, causing API delays and errors.
Likely Cause: Database connection pool exhaustion
Recommended Action: Investigate long-running connections and checkout queues. Review safe pool capacity adjustments.
Reason: Internal model identified database pool saturation.

Evidence:
- Database connections: 100/100
- P95 latency: 1400 ms
- Error rate: 18.5%
- First observation: initial internal state created.
- Internal observation number: 1

SCENARIO: B
Agent: Model-Based Reflex Agent
Summary: High incoming traffic is increasing CPU utilization while the database pool remains healthy.
Likely Cause: Traffic-driven CPU saturation, not DB exhaustion
Recommended Action: Monitor application CPU and consider controlled horizontal scaling.
Reason: Internal model identified high traffic and CPU pressure.

Evidence:
- CPU utilization: 92%
- Requests per minute: 5200
- Database connections: 30/100
- First obse

In [13]:
# VERIFY MODEL-BASED INTERNAL MEMORY

print("BEFORE REPEATED OBSERVATION")

print(
    "Scenario A observation count:",
    model_based_agent.internal_state["A"]["observation_count"]
)

# Observe Scenario A again
repeated_result = model_based_agent.decide(scenarios["A"])

print("\nAFTER REPEATED OBSERVATION")

print(
    "Scenario A observation count:",
    model_based_agent.internal_state["A"]["observation_count"]
)

print("\nPrevious observation evidence:")

for item in repeated_result.evidence:
    print("-", item)

BEFORE REPEATED OBSERVATION
Scenario A observation count: 1

AFTER REPEATED OBSERVATION
Scenario A observation count: 2

Previous observation evidence:
- Database connections: 100/100
- P95 latency: 1400 ms
- Error rate: 18.5%
- Previous observation: incident state was Database connection pool exhaustion
- Metric changes: No metric changes detected.
- Internal observation number: 2


## 3. Goal-Based Agent

The Goal-Based Agent identifies the desired system state
and selects actions to achieve that state.

Main goals:
1. Restore service availability.
2. Reduce application errors.
3. Improve response time.
4. Maintain healthy infrastructure.
5. Recommend safe recovery actions.

In [28]:
# STEP 7.5: GOAL-BASED AGENT

class GoalBasedAgent:

    def decide(self, scenario: IncidentScenario) -> AgentResult:

        metrics = scenario.metrics
        logs = list(scenario.logs)
        log_text = " ".join(logs).lower()

        latency = metrics.get("latency_p95_ms", 0)
        error_rate = metrics.get("error_rate_pct", 0)
        cpu = metrics.get("cpu_pct", 0)
        requests = metrics.get("requests_per_min", 0)
        db_connections = metrics.get("db_connections_in_use", 0)
        db_limit = metrics.get("db_pool_limit", 0)
        availability = metrics.get("availability_pct", 0)

        # GOAL 1: Restore application after startup failure
        if (
            "startup failed" in log_text
            or "missing auth_issuer" in log_text
        ):

            previous = dict(scenario.previous_version)

            deployments = [
                dict(item) for item in scenario.deployment_history
            ]

            failed = next(
                (
                    item for item in reversed(deployments)
                    if "failed" in item.get("status", "").lower()
                ),
                {}
            )

            failed_version = failed.get("version", "unknown")
            healthy_version = previous.get("version", "unknown")

            cause = "Application startup failure due to missing AUTH_ISSUER"

            summary = (
                "The application is unavailable because the latest "
                "deployment failed during startup."
            )

            action = (
                f"Goal: Restore application availability. "
                f"Step 1: Verify the missing AUTH_ISSUER setting. "
                f"Step 2: Compare failed version {failed_version} "
                f"with previous healthy version {healthy_version}. "
                f"Step 3: After operator approval, restore the correct "
                f"configuration or perform a controlled rollback. "
                f"Step 4: Restart and verify application health."
            )

            goals = (
                "Restore application startup.",
                "Restore service availability.",
                "Reduce error rate to an acceptable level.",
            )

            evidence = (
                f"Current availability: {availability}%",
                f"Current error rate: {error_rate}%",
                f"Failed deployment: {failed_version}",
                f"Previous healthy version: {healthy_version}",
                f"Previous configuration: {previous.get('config')}",
                "Missing AUTH_ISSUER confirmed in application logs."
                "The earlier DB transient timeout was marked resolved; "
                "it is historical and inconclusive for the current startup failure."
            )

            verification = (
                "Confirm the application starts successfully.",
                "Verify the AUTH_ISSUER configuration.",
                "Confirm availability has recovered.",
                "Check that the error rate has decreased."
            )

            reason = (
                "Selected the application recovery goal because "
                "the service is unavailable."
            )

        # GOAL 2: Restore database connection health
        elif (
            db_limit > 0
            and db_connections >= db_limit
            and (latency >= 1000 or error_rate >= 5)
        ):

            cause = "Database connection pool exhaustion"

            summary = (
                "The database connection pool is exhausted, "
                "affecting API response time and availability."
            )

            action = (
                "Goal: Restore database connection availability. "
                "Step 1: Identify long-running or leaked connections. "
                "Step 2: Investigate connection wait queues. "
                "Step 3: Review safe pool capacity adjustments. "
                "Step 4: Verify database and API recovery."
            )

            goals = (
                "Bring database connections below the pool limit.",
                "Reduce API latency.",
                "Reduce application error rate.",
                "Restore service availability."
            )

            evidence = (
                f"Database connections: {db_connections}/{db_limit}",
                f"P95 latency: {latency} ms",
                f"Error rate: {error_rate}%",
                f"Availability: {availability}%"
            )

            verification = (
                "Confirm database connections are below the pool limit.",
                "Check database connection wait queues.",
                "Verify API latency has decreased.",
                "Verify application errors have reduced."
            )

            reason = (
                "Selected database recovery goal because the "
                "connection pool is completely occupied."
            )

        # GOAL 3: Reduce CPU pressure
        elif (
            cpu >= 85
            and requests >= 4000
            and db_limit > 0
            and db_connections < db_limit * 0.8
        ):

            cause = "High traffic causing CPU saturation"

            summary = (
                "The system is experiencing high CPU utilization "
                "because of increased incoming traffic."
            )

            action = (
                "Goal: Reduce CPU pressure while maintaining service "
                "availability. Step 1: Identify traffic-heavy endpoints. "
                "Step 2: Review request handling and application workload. "
                "Step 3: Evaluate controlled horizontal scaling. "
                "Step 4: Monitor CPU and response times."
            )

            goals = (
                "Reduce CPU utilization below 80%.",
                "Maintain healthy database connections.",
                "Maintain low application error rates.",
                "Preserve service availability."
            )

            evidence = (
                f"CPU utilization: {cpu}%",
                f"Requests per minute: {requests}",
                f"Database connections: {db_connections}/{db_limit}",
                f"Current error rate: {error_rate}%",
                f"Availability: {availability}%"
            )

            verification = (
                "Confirm CPU utilization is decreasing.",
                "Check that database connections remain healthy.",
                "Verify API latency remains within the target.",
                "Confirm application errors remain low."
            )

            reason = (
                "Selected CPU recovery goal because traffic is high "
                "and database capacity remains available."
            )

        else:

            cause = "No critical incident pattern detected"

            summary = (
                "The available metrics do not indicate a known critical incident."
            )

            action = (
                "Goal: Maintain system health. Continue monitoring "
                "application metrics and logs."
            )

            goals = (
                "Maintain service availability.",
                "Maintain acceptable response times.",
                "Keep error rates low."
            )

            evidence = (
                f"CPU utilization: {cpu}%",
                f"Error rate: {error_rate}%",
                f"Availability: {availability}%"
            )

            verification = (
                "Continue monitoring application health.",
                "Review metrics for emerging problems."
            )

            reason = "Normal monitoring goal selected."

        return AgentResult(
            agent_name="Goal-Based Agent",
            incident_summary=summary,
            evidence=tuple(evidence) + (
                "Selected recovery goals: " + "; ".join(goals),
            ),
            likely_cause=cause,
            recommended_action=action,
            verification_steps=verification,
            limitations=(
                "Uses predefined recovery goals.",
                "Does not automatically execute recovery actions.",
                "Depends on the quality of available metrics and logs."
            ),
            reason=reason
        )


print("Goal-Based Agent class created successfully!")

Goal-Based Agent class created successfully!


In [15]:
# TEST GOAL-BASED AGENT ON ALL THREE SCENARIOS

goal_based_agent = GoalBasedAgent()

goal_based_results = {}

for scenario_id, scenario in scenarios.items():

    result = goal_based_agent.decide(scenario)

    goal_based_results[scenario_id] = result

    print("\n" + "=" * 65)
    print("SCENARIO:", scenario_id)
    print("=" * 65)

    print("Agent:", result.agent_name)
    print("Summary:", result.incident_summary)
    print("Likely Cause:", result.likely_cause)

    print("\nRecommended Goal-Based Action:")
    print(result.recommended_action)

    print("\nEvidence and Goals:")
    for item in result.evidence:
        print("-", item)

    print("\nVerification Steps:")
    for item in result.verification_steps:
        print("-", item)

    print("\nReason:", result.reason)

print("\n" + "=" * 65)
print("GOAL-BASED AGENT TESTING COMPLETED")
print("Total scenarios tested:", len(goal_based_results))
print("=" * 65)


SCENARIO: A
Agent: Goal-Based Agent
Summary: The database connection pool is exhausted, affecting API response time and availability.
Likely Cause: Database connection pool exhaustion

Recommended Goal-Based Action:
Goal: Restore database connection availability. Step 1: Identify long-running or leaked connections. Step 2: Investigate connection wait queues. Step 3: Review safe pool capacity adjustments. Step 4: Verify database and API recovery.

Evidence and Goals:
- Database connections: 100/100
- P95 latency: 1400 ms
- Error rate: 18.5%
- Availability: 81.5%
- Selected recovery goals: Bring database connections below the pool limit.; Reduce API latency.; Reduce application error rate.; Restore service availability.

Verification Steps:
- Confirm database connections are below the pool limit.
- Check database connection wait queues.
- Verify API latency has decreased.
- Verify application errors have reduced.

Reason: Selected database recovery goal because the connection pool is co

## 4. Utility-Based Agent

The Utility-Based Agent evaluates multiple possible
recovery actions and selects the action with the highest
utility score.

It considers:
1. Expected service recovery benefit.
2. Probability of success.
3. Recovery speed.
4. Operational safety.
5. Implementation cost.

The selected action is the one with the highest
calculated utility score.

In [29]:
# STEP 7.6: UTILITY-BASED AGENT

class UtilityBasedAgent:

    def __init__(self):

        # Weights used for utility calculation
        self.weights = {
            "benefit": 0.30,
            "success": 0.25,
            "speed": 0.20,
            "safety": 0.15,
            "cost_efficiency": 0.10
        }

    def calculate_utility(self, action):

        score = sum(
            self.weights[key] * action[key]
            for key in self.weights
        )

        return round(score, 2)

    def decide(self, scenario: IncidentScenario) -> AgentResult:

        metrics = scenario.metrics
        logs = list(scenario.logs)
        log_text = " ".join(logs).lower()

        latency = metrics.get("latency_p95_ms", 0)
        error_rate = metrics.get("error_rate_pct", 0)
        cpu = metrics.get("cpu_pct", 0)
        requests = metrics.get("requests_per_min", 0)
        db_connections = metrics.get("db_connections_in_use", 0)
        db_limit = metrics.get("db_pool_limit", 0)
        availability = metrics.get("availability_pct", 0)

        # Identify incident and generate action alternatives

        if (
            "startup failed" in log_text
            or "missing auth_issuer" in log_text
        ):

            cause = "Application startup failure due to missing AUTH_ISSUER"

            previous = dict(scenario.previous_version)

            deployments = [
                dict(item) for item in scenario.deployment_history
            ]

            failed = next(
                (
                    item for item in reversed(deployments)
                    if "failed" in item.get("status", "").lower()
                ),
                {}
            )

            failed_version = failed.get("version", "unknown")
            healthy_version = previous.get("version", "unknown")

            actions = [
                {
                    "name": "Restore missing AUTH_ISSUER configuration",
                    "benefit": 10,
                    "success": 9,
                    "speed": 9,
                    "safety": 9,
                    "cost_efficiency": 10
                },
                {
                    "name": "Controlled rollback to previous healthy version",
                    "benefit": 10,
                    "success": 9,
                    "speed": 8,
                    "safety": 8,
                    "cost_efficiency": 7
                },
                {
                    "name": "Restart application without configuration correction",
                    "benefit": 2,
                    "success": 2,
                    "speed": 8,
                    "safety": 3,
                    "cost_efficiency": 9
                }
            ]

            summary = (
                f"Deployment {failed_version} failed because "
                "the AUTH_ISSUER setting is missing."
            )

            verification = (
                "Verify AUTH_ISSUER configuration.",
                "Confirm application startup.",
                "Check application availability.",
                "Verify the error rate decreases."
            )

            evidence = [
                f"Failed version: {failed_version}",
                f"Previous healthy version: {healthy_version}",
                f"Previous configuration: {previous.get('config')}",
                f"Availability: {availability}%",
                f"Error rate: {error_rate}%"
                "The earlier DB transient timeout was marked resolved; "
                "it is historical and inconclusive for the current startup failure."
            ]

        elif (
            db_limit > 0
            and db_connections >= db_limit
            and (latency >= 1000 or error_rate >= 5)
        ):

            cause = "Database connection pool exhaustion"

            actions = [
                {
                    "name": "Investigate and release stuck database connections",
                    "benefit": 10,
                    "success": 9,
                    "speed": 8,
                    "safety": 9,
                    "cost_efficiency": 9
                },
                {
                    "name": "Increase database connection pool capacity after review",
                    "benefit": 8,
                    "success": 8,
                    "speed": 7,
                    "safety": 7,
                    "cost_efficiency": 7
                },
                {
                    "name": "Restart all application services immediately",
                    "benefit": 6,
                    "success": 5,
                    "speed": 8,
                    "safety": 3,
                    "cost_efficiency": 6
                }
            ]

            summary = (
                "The database connection pool is fully occupied, "
                "causing API delays and errors."
            )

            verification = (
                "Check database connection utilization.",
                "Verify connection wait queues decrease.",
                "Verify API latency improves.",
                "Confirm application error rate decreases."
            )

            evidence = [
                f"Database connections: {db_connections}/{db_limit}",
                f"P95 latency: {latency} ms",
                f"Error rate: {error_rate}%",
                f"Availability: {availability}%"
            ]

        elif (
            cpu >= 85
            and requests >= 4000
            and db_limit > 0
            and db_connections < db_limit * 0.8
        ):

            cause = "High traffic causing CPU saturation"

            actions = [
                {
                    "name": "Controlled horizontal application scaling",
                    "benefit": 10,
                    "success": 9,
                    "speed": 8,
                    "safety": 9,
                    "cost_efficiency": 7
                },
                {
                    "name": "Optimize CPU-intensive application requests",
                    "benefit": 8,
                    "success": 8,
                    "speed": 6,
                    "safety": 9,
                    "cost_efficiency": 9
                },
                {
                    "name": "Immediately restart all services",
                    "benefit": 3,
                    "success": 3,
                    "speed": 7,
                    "safety": 2,
                    "cost_efficiency": 5
                }
            ]

            summary = (
                "High incoming traffic is causing CPU saturation "
                "while database connections remain healthy."
            )

            verification = (
                "Monitor CPU utilization.",
                "Check incoming request volume.",
                "Confirm database connections remain healthy.",
                "Verify API latency and error rate."
            )

            evidence = [
                f"CPU utilization: {cpu}%",
                f"Requests per minute: {requests}",
                f"Database connections: {db_connections}/{db_limit}",
                f"Error rate: {error_rate}%"
            ]

        else:

            cause = "No known critical incident pattern detected"

            actions = [
                {
                    "name": "Continue monitoring system health",
                    "benefit": 9,
                    "success": 10,
                    "speed": 10,
                    "safety": 10,
                    "cost_efficiency": 10
                },
                {
                    "name": "Perform unnecessary immediate restart",
                    "benefit": 2,
                    "success": 4,
                    "speed": 8,
                    "safety": 3,
                    "cost_efficiency": 4
                }
            ]

            summary = "No known critical incident was identified."

            verification = (
                "Continue monitoring application metrics.",
                "Review logs for new incidents."
            )

            evidence = [
                f"CPU utilization: {cpu}%",
                f"Error rate: {error_rate}%",
                f"Availability: {availability}%"
            ]

        # Calculate utility of every possible action

        for action in actions:
            action["utility_score"] = self.calculate_utility(action)

        # Select action with highest utility

        best_action = max(
            actions,
            key=lambda action: action["utility_score"]
        )

        # Prepare action evaluation evidence

        action_scores = [
            (
                f"{action['name']}: "
                f"Utility score = {action['utility_score']}/10"
            )
            for action in actions
        ]

        evidence.extend(action_scores)

        evidence.append(
            f"Selected action: {best_action['name']}"
        )

        evidence.append(
            f"Highest utility score: {best_action['utility_score']}/10"
        )

        # Return standardized shared result

        return AgentResult(
            agent_name="Utility-Based Agent",
            incident_summary=summary,
            evidence=tuple(evidence),
            likely_cause=cause,
            recommended_action=(
                f"{best_action['name']}. "
                "Obtain operator approval before executing any "
                "production change."
            ),
            verification_steps=verification,
            limitations=(
                "Utility weights are predefined.",
                "Scores are estimates and not measured recovery probabilities.",
                "Does not automatically execute recovery actions."
            ),
            reason=(
                "Evaluated multiple recovery alternatives using "
                "weighted utility scores and selected the "
                "highest-scoring action."
            )
        )


print("Utility-Based Agent class created successfully!")

Utility-Based Agent class created successfully!


In [17]:
# TEST UTILITY-BASED AGENT ON ALL THREE SCENARIOS

utility_agent = UtilityBasedAgent()

utility_results = {}

for scenario_id, scenario in scenarios.items():

    result = utility_agent.decide(scenario)

    utility_results[scenario_id] = result

    print("\n" + "=" * 70)
    print("SCENARIO:", scenario_id)
    print("=" * 70)

    print("Agent:", result.agent_name)
    print("Summary:", result.incident_summary)
    print("Likely Cause:", result.likely_cause)

    print("\nACTION EVALUATION:")

    for item in result.evidence:
        if "Utility score" in item:
            print("-", item)

    print("\nSELECTED ACTION:")
    print(result.recommended_action)

    print("\nREASON:")
    print(result.reason)

    print("\nVERIFICATION:")
    for item in result.verification_steps:
        print("-", item)

print("\n" + "=" * 70)
print("UTILITY-BASED AGENT TESTING COMPLETED")
print("Total scenarios tested:", len(utility_results))
print("=" * 70)


SCENARIO: A
Agent: Utility-Based Agent
Summary: The database connection pool is fully occupied, causing API delays and errors.
Likely Cause: Database connection pool exhaustion

ACTION EVALUATION:
- Investigate and release stuck database connections: Utility score = 9.1/10
- Increase database connection pool capacity after review: Utility score = 7.55/10
- Restart all application services immediately: Utility score = 5.7/10

SELECTED ACTION:
Investigate and release stuck database connections. Obtain operator approval before executing any production change.

REASON:
Evaluated multiple recovery alternatives using weighted utility scores and selected the highest-scoring action.

VERIFICATION:
- Check database connection utilization.
- Verify connection wait queues decrease.
- Verify API latency improves.
- Confirm application error rate decreases.

SCENARIO: B
Agent: Utility-Based Agent
Summary: High incoming traffic is causing CPU saturation while database connections remain healthy.
Li

## 5. Learning Agent

The Learning Agent improves its future decisions using
feedback from previous incident outcomes.

It maintains a persistent experience database.

Learning process:
1. Analyze an incident.
2. Recommend a recovery action.
3. Receive feedback on the action's outcome.
4. Update the experience database.
5. Use previous feedback to improve future recommendations.

The experience database is stored as a JSON file.

In [30]:
# STEP 7.7: LEARNING AGENT

import json
from pathlib import Path

class LearningAgent:

    def __init__(self, experience_path=None):

        self.experience_path = (
            Path(experience_path)
            if experience_path
            else PROJECT_ROOT / "data" / "learning_agent_experience.json"
        )

        self.experience_path.parent.mkdir(parents=True, exist_ok=True)

        self.experience = {
            "schema_version": 1,
            "feedback": {}
        }

        # Load previous learning experiences
        if self.experience_path.exists():
            self.experience = json.loads(
                self.experience_path.read_text(encoding="utf-8")
            )

        print("Learning Agent initialized.")
        print("Experience file:", self.experience_path)
        print(
            "Existing feedback categories:",
            list(self.experience["feedback"].keys())
        )

    def _incident_details(self, scenario):

        metrics = scenario.metrics
        logs = " ".join(scenario.logs).lower()

        latency = metrics.get("latency_p95_ms", 0)
        error = metrics.get("error_rate_pct", 0)
        cpu = metrics.get("cpu_pct", 0)
        requests = metrics.get("requests_per_min", 0)
        db = metrics.get("db_connections_in_use", 0)
        limit = metrics.get("db_pool_limit", 0)

        # Identify incident category

        if "startup failed" in logs or "missing auth_issuer" in logs:

            category = "configuration"

            cause = "Startup failure due to missing AUTH_ISSUER"

            summary = (
                "Application startup failed after a configuration change."
            )

            previous = dict(scenario.previous_version)

            failed = next(
                (
                    dict(item) for item in reversed(scenario.deployment_history)
                    if "failed" in item.get("status", "").lower()
                ),
                {}
            )

            evidence = [
                f"Failed deployment: {failed.get('version')}",
                f"Previous healthy version: {previous.get('version')}",
                f"Previous configuration: {previous.get('config')}"
                "The earlier DB transient timeout was marked resolved; "
                "it is historical and inconclusive for the current startup failure."
            ]

            actions = [
                ("Restore missing AUTH_ISSUER configuration", 9.4),
                ("Controlled rollback to previous healthy version", 8.75),
                ("Restart application without configuration correction", 4.05)
            ]

            verification = (
                "Verify AUTH_ISSUER configuration.",
                "Confirm successful application startup.",
                "Check service availability and error rate."
            )

        elif (
            limit > 0 and db >= limit
            and (latency >= 1000 or error >= 5)
        ):

            category = "database"

            cause = "Database connection pool exhaustion"

            summary = "Database connections are fully occupied."

            evidence = [
                f"Database connections: {db}/{limit}",
                f"P95 latency: {latency} ms",
                f"Error rate: {error}%"
            ]

            actions = [
                ("Investigate and release stuck database connections", 9.1),
                ("Increase database connection pool capacity after review", 7.55),
                ("Restart all application services immediately", 5.7)
            ]

            verification = (
                "Check database connection utilization.",
                "Verify connection wait queues decrease.",
                "Confirm API latency and errors improve."
            )

        elif cpu >= 85 and requests >= 4000 and limit > 0 and db < limit * 0.8:

            category = "traffic"

            cause = "High traffic causing CPU saturation"

            summary = "High incoming traffic is increasing CPU utilization."

            evidence = [
                f"CPU utilization: {cpu}%",
                f"Requests per minute: {requests}",
                f"Database connections: {db}/{limit}"
            ]

            actions = [
                ("Controlled horizontal application scaling", 8.9),
                ("Optimize CPU-intensive application requests", 7.85),
                ("Immediately restart all services", 3.85)
            ]

            verification = (
                "Monitor CPU utilization.",
                "Check incoming traffic.",
                "Verify database connections remain healthy."
            )

        else:

            category = "uncertain"

            cause = "Unknown incident"

            summary = "Insufficient evidence to classify the incident."

            evidence = ["Additional evidence is required."]

            actions = [
                ("Collect additional monitoring evidence", 8.0)
            ]

            verification = (
                "Collect additional metrics and logs.",
                "Review the incident again."
            )

        return category, cause, summary, evidence, actions, verification

    def decide(self, scenario):

        category, cause, summary, evidence, actions, verification = (
            self._incident_details(scenario)
        )

        category_feedback = self.experience["feedback"].get(category, {})

        evaluated_actions = []

        for action_name, base_score in actions:

            record = category_feedback.get(action_name, {})

            successes = record.get("successes", 0)
            failures = record.get("failures", 0)

            attempts = successes + failures

            # Start with the predefined score.
            # Adjust it using previous feedback when available.

            if attempts > 0:

                success_rate = successes / attempts

                learning_score = (
                    0.7 * base_score
                    + 0.3 * (success_rate * 10)
                )

            else:

                learning_score = base_score

            evaluated_actions.append({
                "name": action_name,
                "base_score": base_score,
                "learning_score": round(learning_score, 2),
                "feedback_count": attempts
            })

        # Choose the highest-scoring action

        best_action = max(
            evaluated_actions,
            key=lambda item: item["learning_score"]
        )

        learning_evidence = list(evidence)

        for item in evaluated_actions:

            learning_evidence.append(
                f"{item['name']}: "
                f"Base score={item['base_score']}, "
                f"Learning score={item['learning_score']}, "
                f"Feedback records={item['feedback_count']}"
            )

        if any(item["feedback_count"] > 0 for item in evaluated_actions):

            reason = (
                "Recommendation selected using predefined scores "
                "adjusted by previous feedback."
            )

        else:

            reason = (
                "No previous feedback available. "
                "Initial recommendation uses predefined scores."
            )

        learning_evidence.append(
            f"Selected action: {best_action['name']}"
        )

        learning_evidence.append(
            f"Selected learning score: {best_action['learning_score']}/10"
        )

        return AgentResult(
            agent_name="Learning Agent",
            incident_summary=summary,
            evidence=tuple(learning_evidence),
            likely_cause=cause,
            recommended_action=(
                best_action["name"]
                + ". Obtain operator approval before production changes."
            ),
            verification_steps=verification,
            limitations=(
                "Learning depends on the quality of feedback.",
                "Uses a simple feedback-based scoring mechanism.",
                "Feedback scores do not guarantee real-world success.",
                "Does not automatically execute recovery actions."
            ),
            reason=reason
        )

    # FEEDBACK HOOK

    def record_feedback(self, scenario, action_name, successful, notes=""):

        if not isinstance(successful, bool):
            raise ValueError("successful must be True or False")

        category, _, _, _, actions, _ = self._incident_details(scenario)

        valid_actions = [name for name, _ in actions]

        if action_name not in valid_actions:
            raise ValueError("Unknown action for this incident category.")

        category_feedback = self.experience["feedback"].setdefault(
            category, {}
        )

        record = category_feedback.setdefault(
            action_name,
            {
                "successes": 0,
                "failures": 0,
                "notes": []
            }
        )

        if successful:
            record["successes"] += 1
        else:
            record["failures"] += 1

        if notes:
            record["notes"].append(notes)

        # Save learning experience permanently

        self.experience_path.write_text(
            json.dumps(self.experience, indent=4),
            encoding="utf-8"
        )

        print("Feedback successfully recorded!")
        print("Incident category:", category)
        print("Action:", action_name)
        print("Successful:", successful)
        print("Experience database updated:", self.experience_path)

In [19]:
# TEST LEARNING AGENT: INITIAL RECOMMENDATIONS

learning_agent = LearningAgent()

learning_results = {}

for scenario_id, scenario in scenarios.items():

    result = learning_agent.decide(scenario)

    learning_results[scenario_id] = result

    print("\n" + "=" * 65)
    print("SCENARIO:", scenario_id)
    print("=" * 65)

    print("Likely Cause:", result.likely_cause)
    print("Recommended Action:", result.recommended_action)

    print("\nLearning Evidence:")

    for item in result.evidence:
        print("-", item)

    print("\nReason:", result.reason)

print("\n" + "=" * 65)
print("INITIAL LEARNING AGENT TEST COMPLETED")
print("Scenarios tested:", len(learning_results))
print("Experience file:", learning_agent.experience_path)
print("=" * 65)

Learning Agent initialized.
Experience file: C:\Users\CHAKRITHA\Documents\Python Files- BITS\BITS material\Agentic AI\Assignment1\data\learning_agent_experience.json
Existing feedback categories: []

SCENARIO: A
Likely Cause: Database connection pool exhaustion
Recommended Action: Investigate and release stuck database connections. Obtain operator approval before production changes.

Learning Evidence:
- Database connections: 100/100
- P95 latency: 1400 ms
- Error rate: 18.5%
- Investigate and release stuck database connections: Base score=9.1, Learning score=9.1, Feedback records=0
- Increase database connection pool capacity after review: Base score=7.55, Learning score=7.55, Feedback records=0
- Restart all application services immediately: Base score=5.7, Learning score=5.7, Feedback records=0
- Selected action: Investigate and release stuck database connections
- Selected learning score: 9.1/10

Reason: No previous feedback available. Initial recommendation uses predefined scores.

In [20]:
# BEFORE AND AFTER LEARNING DEMONSTRATION

scenario_a = scenarios["A"]

print("=" * 65)
print("BEFORE LEARNING")
print("=" * 65)

before_result = learning_agent.decide(scenario_a)

print("Recommended Action:")
print(before_result.recommended_action)

# Provide demonstration feedback.
# No actual production action is executed.

learning_agent.record_feedback(
    scenario=scenario_a,
    action_name="Investigate and release stuck database connections",
    successful=False,
    notes="SIMULATED demonstration feedback only."
)

learning_agent.record_feedback(
    scenario=scenario_a,
    action_name="Increase database connection pool capacity after review",
    successful=True,
    notes="SIMULATED demonstration feedback only."
)

print("\n" + "=" * 65)
print("AFTER LEARNING")
print("=" * 65)

after_result = learning_agent.decide(scenario_a)

print("New Recommended Action:")
print(after_result.recommended_action)

print("\nUpdated Learning Evidence:")

for item in after_result.evidence:
    print("-", item)

print("\n" + "=" * 65)
print("BEFORE AND AFTER COMPARISON")
print("=" * 65)

print("Before:", before_result.recommended_action)
print("After :", after_result.recommended_action)

print("\nExperience database:")
print(learning_agent.experience_path)

print("\nLEARNING DEMONSTRATION COMPLETED")

BEFORE LEARNING
Recommended Action:
Investigate and release stuck database connections. Obtain operator approval before production changes.
Feedback successfully recorded!
Incident category: database
Action: Investigate and release stuck database connections
Successful: False
Experience database updated: C:\Users\CHAKRITHA\Documents\Python Files- BITS\BITS material\Agentic AI\Assignment1\data\learning_agent_experience.json
Feedback successfully recorded!
Incident category: database
Action: Increase database connection pool capacity after review
Successful: True
Experience database updated: C:\Users\CHAKRITHA\Documents\Python Files- BITS\BITS material\Agentic AI\Assignment1\data\learning_agent_experience.json

AFTER LEARNING
New Recommended Action:
Increase database connection pool capacity after review. Obtain operator approval before production changes.

Updated Learning Evidence:
- Database connections: 100/100
- P95 latency: 1400 ms
- Error rate: 18.5%
- Investigate and release stuc

In [21]:
# VERIFY PERSISTENT LEARNING DATABASE

import json

experience_file = (
    PROJECT_ROOT / "data" / "learning_agent_experience.json"
)

print("File exists:", experience_file.exists())

saved_data = json.loads(
    experience_file.read_text(encoding="utf-8")
)

print("\nLEARNING CATEGORIES:")

for category, actions in saved_data["feedback"].items():
    print("\nCategory:", category)

    for action, record in actions.items():
        print("Action:", action)
        print("Successes:", record["successes"])
        print("Failures:", record["failures"])

print("\nLEARNING DATABASE VERIFICATION COMPLETED")

File exists: True

LEARNING CATEGORIES:

Category: database
Action: Investigate and release stuck database connections
Successes: 0
Failures: 1
Action: Increase database connection pool capacity after review
Successes: 1
Failures: 0

LEARNING DATABASE VERIFICATION COMPLETED


# 6. Classical Agent Registry

This section combines all five classical agents into
one callable registry.

Each agent accepts the same shared incident scenario
and returns the common AgentResult structure.

In [31]:
# CREATE FRESH AGENT INSTANCES FOR FINAL COMPARISON

from incd_assist.contracts import AgentResult

# Separate clean learning file for the final experiment.
# Your earlier learning evidence remains preserved.

baseline_learning_file = (
    PROJECT_ROOT / "data" / "learning_agent_experience_baseline.json"
)

# Reset only this dedicated baseline experiment file.
if baseline_learning_file.exists():
    baseline_learning_file.unlink()

AGENT_REGISTRY = {
    "Simple Reflex Agent": SimpleReflexAgent(),
    "Model-Based Reflex Agent": ModelBasedReflexAgent(),
    "Goal-Based Agent": GoalBasedAgent(),
    "Utility-Based Agent": UtilityBasedAgent(),
    "Learning Agent": LearningAgent(
        experience_path=baseline_learning_file
    )
}

# Verify callable interface

for name, agent in AGENT_REGISTRY.items():
    assert callable(agent.decide), f"{name} has no decide method"

print("=" * 60)
print("CLASSICAL AGENT REGISTRY CREATED")
print("=" * 60)

for name in AGENT_REGISTRY:
    print("PASS:", name)

print("\nTotal classical agents:", len(AGENT_REGISTRY))
print("All agents support decide(scenario).")

Learning Agent initialized.
Experience file: C:\Users\CHAKRITHA\Documents\Python Files- BITS\BITS material\Agentic AI\Assignment1\data\learning_agent_experience_baseline.json
Existing feedback categories: []
CLASSICAL AGENT REGISTRY CREATED
PASS: Simple Reflex Agent
PASS: Model-Based Reflex Agent
PASS: Goal-Based Agent
PASS: Utility-Based Agent
PASS: Learning Agent

Total classical agents: 5
All agents support decide(scenario).


In [32]:
# STEP 8.2: INSPECT OFFICIAL EXPECTED RESULTS

import json

expected_file = PROJECT_ROOT / "data" / "expected_results.json"

with open(expected_file, "r", encoding="utf-8") as file:
    expected_results = json.load(file)

print("=" * 60)
print("OFFICIAL EXPECTED RESULTS")
print("=" * 60)

print(json.dumps(expected_results, indent=4))

print("\nExpected results loaded successfully!")

OFFICIAL EXPECTED RESULTS
{
    "A": {
        "category": "critical",
        "likely_cause": "DB connection pool exhaustion",
        "safe_action": "Investigate leaked/long-running DB connections; limit load and restore pool capacity cautiously.",
        "verification": "DB checkout queue and request error/latency rates recover."
    },
    "B": {
        "category": "normal high load",
        "likely_cause": "Traffic-driven CPU saturation, not DB exhaustion",
        "safe_action": "Monitor or scale application capacity; do not roll back or change DB pool blindly.",
        "verification": "CPU and latency stabilize while availability remains high."
    },
    "C": {
        "category": "edge/ambiguous failure",
        "likely_cause": "Missing startup setting after config change; stale DB warning is inconclusive",
        "safe_action": "Verify current startup logs and configuration; restore last-known-good config/version if confirmed.",
        "verification": "Startup and heal

## 7. Final Classical Agent Comparison

All five classical agents are evaluated against
the same three shared incident scenarios.

Total experiments: 5 agents × 3 scenarios = 15 results.

Official expected results are used only for evaluation,
never as agent input.

In [34]:
# STEP 8.3: RUN ALL FIVE AGENTS ON ALL THREE SCENARIOS

import json

from pathlib import Path

classical_results = {}

for scenario_id, scenario in scenarios.items():

    classical_results[scenario_id] = {}

    print("\n" + "=" * 70)
    print("SCENARIO:", scenario_id, "-", scenario.label)
    print("=" * 70)

    for agent_name, agent in AGENT_REGISTRY.items():

        result = agent.decide(scenario)

        classical_results[scenario_id][agent_name] = result

        print("\nAgent:", agent_name)
        print("Likely Cause:", result.likely_cause)
        print("Recommended Action:", result.recommended_action)

# Convert AgentResult objects into JSON-compatible dictionaries

results_to_save = {}

for scenario_id, agent_results in classical_results.items():

    results_to_save[scenario_id] = {}

    for agent_name, result in agent_results.items():

        results_to_save[scenario_id][agent_name] = {
            "agent_name": result.agent_name,
            "incident_summary": result.incident_summary,
            "likely_cause": result.likely_cause,
            "recommended_action": result.recommended_action,
            "evidence": list(result.evidence),
            "verification_steps": list(result.verification_steps),
            "limitations": list(result.limitations),
            "reason": result.reason
        }

# Save results

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

results_file = OUTPUT_DIR / "chakritha_classical_results.json"

with open(results_file, "w", encoding="utf-8") as file:
    json.dump(results_to_save, file, indent=4)

total_results = sum(
    len(agent_results)
    for agent_results in classical_results.values()
)

print("\n" + "=" * 70)
print("CLASSICAL AGENT EXPERIMENT COMPLETED")
print("=" * 70)

print("Scenarios:", len(classical_results))
print("Agents:", len(AGENT_REGISTRY))
print("Total results:", total_results)
print("Expected results: 15")
print("Results match expected count:", total_results == 15)

print("\nSaved file:")
print(results_file)


SCENARIO: A - API latency and DB pool exhaustion

Agent: Simple Reflex Agent
Likely Cause: Database connection pool exhaustion
Recommended Action: Investigate long-running database connections, connection wait queues and pool capacity. Review safe capacity adjustments.

Agent: Model-Based Reflex Agent
Likely Cause: Database connection pool exhaustion
Recommended Action: Investigate long-running connections and checkout queues. Review safe pool capacity adjustments.

Agent: Goal-Based Agent
Likely Cause: Database connection pool exhaustion
Recommended Action: Goal: Restore database connection availability. Step 1: Identify long-running or leaked connections. Step 2: Investigate connection wait queues. Step 3: Review safe pool capacity adjustments. Step 4: Verify database and API recovery.

Agent: Utility-Based Agent
Likely Cause: Database connection pool exhaustion
Recommended Action: Investigate and release stuck database connections. Obtain operator approval before executing any prod

## 8. Classical Agent Evaluation

Evaluate the five agents against the official expected
incident classifications for scenarios A, B and C.

Check classification accuracy and Scenario C historical
database warning evidence.

In [35]:
# STEP 8.4: EVALUATE ALL 15 CLASSICAL RESULTS

import pandas as pd

def check_classification(scenario_id, cause):

    cause = cause.lower()

    if scenario_id == "A":
        return (
            ("database" in cause or "db" in cause)
            and ("connection" in cause or "pool" in cause)
            and ("exhaust" in cause or "occupied" in cause)
        )

    elif scenario_id == "B":
        return (
            "traffic" in cause
            and "cpu" in cause
            and ("saturat" in cause or "utilization" in cause)
        )

    elif scenario_id == "C":
        return (
            "startup" in cause
            and ("auth_issuer" in cause or "configuration" in cause)
        )

evaluation_rows = []

for scenario_id, agent_results in classical_results.items():

    expected_cause = expected_results[scenario_id]["likely_cause"]

    for agent_name, result in agent_results.items():

        matched = check_classification(
            scenario_id,
            result.likely_cause
        )

        evidence_text = " ".join(result.evidence).lower()

        # Specifically check whether the historical DB warning
        # is acknowledged for Scenario C.

        historical_warning_acknowledged = (
            "earlier db timeout" in evidence_text
            or "transient timeout" in evidence_text
            or "historical" in evidence_text
        ) if scenario_id == "C" else "N/A"

        evaluation_rows.append({
            "Scenario": scenario_id,
            "Agent": agent_name,
            "Expected Cause": expected_cause,
            "Predicted Cause": result.likely_cause,
            "Classification Match": matched,
            "Historical DB Warning Acknowledged": historical_warning_acknowledged
        })

evaluation_df = pd.DataFrame(evaluation_rows)

print("=" * 80)
print("CLASSICAL AGENT EVALUATION")
print("=" * 80)

display(evaluation_df)

correct = evaluation_df["Classification Match"].sum()
total = len(evaluation_df)

print("\nCorrect classifications:", correct, "/", total)
print("Classification accuracy:", round(correct / total * 100, 2), "%")

print("\nSCENARIO C HISTORICAL EVIDENCE REVIEW")

c_review = evaluation_df[
    evaluation_df["Scenario"] == "C"
]

display(
    c_review[
        ["Agent", "Classification Match",
         "Historical DB Warning Acknowledged"]
    ]
)

# Save evaluation

evaluation_file = OUTPUT_DIR / "chakritha_classical_evaluation.csv"

evaluation_df.to_csv(evaluation_file, index=False)

print("\nEvaluation saved:")
print(evaluation_file)

CLASSICAL AGENT EVALUATION


,Scenario,Agent,Expected Cause,Predicted Cause,Classification Match,Historical DB Warning Acknowledged
0,A,Simple Reflex Agent,DB connection pool exhaustion,Database connection pool exhaustion,True,N/A
1,A,Model-Based Reflex Agent,DB connection pool exhaustion,Database connection pool exhaustion,True,N/A
2,A,Goal-Based Agent,DB connection pool exhaustion,Database connection pool exhaustion,True,N/A
3,A,Utility-Based Agent,DB connection pool exhaustion,Database connection pool exhaustion,True,N/A
4,A,Learning Agent,DB connection pool exhaustion,Database connection pool exhaustion,True,N/A
5,B,Simple Reflex Agent,"Traffic-driven CPU saturation, not DB exhaustion",High traffic causing CPU saturation,True,N/A
6,B,Model-Based Reflex Agent,"Traffic-driven CPU saturation, not DB exhaustion","Traffic-driven CPU saturation, not DB exhaustion",True,N/A
7,B,Goal-Based Agent,"Traffic-driven CPU saturation, not DB exhaustion",High traffic causing CPU saturation,True,N/A
8,B,Utility-Based Agent,"Traffic-driven CPU saturation, not DB exhaustion",High traffic causing CPU saturation,True,N/A
9,B,Learning Agent,"Traffic-driven CPU saturation, not DB exhaustion",High traffic causing CPU saturation,True,N/A



Correct classifications: 15 / 15
Classification accuracy: 100.0 %

SCENARIO C HISTORICAL EVIDENCE REVIEW


,Agent,Classification Match,Historical DB Warning Acknowledged
10,Simple Reflex Agent,True,True
11,Model-Based Reflex Agent,True,True
12,Goal-Based Agent,True,True
13,Utility-Based Agent,True,True
14,Learning Agent,True,True



Evaluation saved:
C:\Users\CHAKRITHA\Documents\Python Files- BITS\BITS material\Agentic AI\Assignment1\outputs\chakritha_classical_evaluation.csv


In [36]:
# INSPECT SHARED AGENT INTERFACE

import inspect
from incd_assist.contracts import Agent, AgentResult

print("AGENT INTERFACE")
print(inspect.getsource(Agent))

print("\nAGENT RESULT STRUCTURE")
print(inspect.signature(AgentResult))

print("\nPROJECT PYTHON FILES")

for file in sorted((PROJECT_ROOT / "incd_assist").rglob("*.py")):
    print(file.relative_to(PROJECT_ROOT))

AGENT INTERFACE
class Agent(Protocol):
    def decide(self, scenario: IncidentScenario) -> AgentResult: ...


AGENT RESULT STRUCTURE
(agent_name: str, incident_summary: str, evidence: tuple[str, ...], likely_cause: str, recommended_action: str, verification_steps: tuple[str, ...], limitations: tuple[str, ...], reason: str) -> None

PROJECT PYTHON FILES
incd_assist\__init__.py
incd_assist\contracts.py
incd_assist\scenarios.py
incd_assist\tools.py


## 9. Export Classical Agents for Team Integration

Export all five classical agent classes into a reusable
Python module inside the shared incd_assist package.

In [38]:
# STEP 8.7: EXPORT FIVE AGENTS INTO A PYTHON MODULE

import json
from pathlib import Path

notebook_path = (
    PROJECT_ROOT / "notebooks" / "chakritha_classical_agents.ipynb"
)

assert notebook_path.exists(), "Save the notebook before running this cell."

with open(notebook_path, "r", encoding="utf-8") as file:
    notebook_data = json.load(file)

code_cells = [
    "".join(cell["source"])
    if isinstance(cell["source"], list)
    else cell["source"]
    for cell in notebook_data["cells"]
    if cell["cell_type"] == "code"
]

agent_markers = [
    "class SimpleReflexAgent:",
    "class ModelBasedReflexAgent:",
    "class GoalBasedAgent:",
    "class UtilityBasedAgent:",
    "class LearningAgent:"
]

agent_sources = []

for marker in agent_markers:

    matches = [
        source for source in code_cells
        if marker in source
    ]

    assert len(matches) == 1, (
        f"Expected exactly one code cell containing {marker}. "
        f"Found: {len(matches)}"
    )

    agent_sources.append(matches[0])

header = '''"""
Chakritha's five classical agents for incident analysis.
"""

import json
from pathlib import Path

from incd_assist.contracts import Agent, AgentResult, IncidentScenario

PROJECT_ROOT = Path(__file__).resolve().parent.parent

'''

factory = '''

def create_classical_agents(experience_path=None) -> dict[str, Agent]:
    """Create fresh instances of all five classical agents."""

    return {
        "Simple Reflex Agent": SimpleReflexAgent(),
        "Model-Based Reflex Agent": ModelBasedReflexAgent(),
        "Goal-Based Agent": GoalBasedAgent(),
        "Utility-Based Agent": UtilityBasedAgent(),
        "Learning Agent": LearningAgent(
            experience_path=experience_path
        ),
    }
'''

module_content = (
    header
    + "\n\n".join(agent_sources)
    + factory
)

module_path = PROJECT_ROOT / "incd_assist" / "classical_agents.py"

# Check Python syntax before writing
compile(module_content, str(module_path), "exec")

module_path.write_text(
    module_content,
    encoding="utf-8"
)

print("CLASSICAL AGENT MODULE CREATED")
print("Location:", module_path)
print("Agents exported:", len(agent_sources))
print("Python syntax: PASS")

CLASSICAL AGENT MODULE CREATED
Location: C:\Users\CHAKRITHA\Documents\Python Files- BITS\BITS material\Agentic AI\Assignment1\incd_assist\classical_agents.py
Agents exported: 5
Python syntax: PASS


In [39]:
# VERIFY IMPORT AND EXECUTION OF ALL FIVE EXPORTED AGENTS

import importlib
from incd_assist.contracts import AgentResult

import incd_assist.classical_agents as classical_module

importlib.invalidate_caches()
classical_module = importlib.reload(classical_module)

handoff_agents = classical_module.create_classical_agents()

total_results = 0

for scenario_id, scenario in scenarios.items():

    for agent_name, agent in handoff_agents.items():

        result = agent.decide(scenario)

        assert isinstance(result, AgentResult), (
            f"Invalid result from {agent_name}"
        )

        total_results += 1

print("=" * 60)
print("INTEGRATION MODULE VERIFICATION")
print("=" * 60)

for name in handoff_agents:
    print("PASS:", name)

print("\nTotal agents:", len(handoff_agents))
print("Total test results:", total_results)
print("Expected results: 15")
print("Integration module: PASS")

Simple Reflex Agent class created successfully!
Model-Based Reflex Agent class created successfully!
Goal-Based Agent class created successfully!
Utility-Based Agent class created successfully!
Simple Reflex Agent class created successfully!
Model-Based Reflex Agent class created successfully!
Goal-Based Agent class created successfully!
Utility-Based Agent class created successfully!
Learning Agent initialized.
Experience file: C:\Users\CHAKRITHA\Documents\Python Files- BITS\BITS material\Agentic AI\Assignment1\data\learning_agent_experience.json
Existing feedback categories: ['database']
INTEGRATION MODULE VERIFICATION
PASS: Simple Reflex Agent
PASS: Model-Based Reflex Agent
PASS: Goal-Based Agent
PASS: Utility-Based Agent
PASS: Learning Agent

Total agents: 5
Total test results: 15
Expected results: 15
Integration module: PASS


In [40]:
# CLEAN MODULE AND SEPARATE LEARNING DATABASE

import ast

module_path = PROJECT_ROOT / "incd_assist" / "classical_agents.py"

source = module_path.read_text(encoding="utf-8")

# Use a separate, clean persistent database for Vishruth's integration
old = '''"Learning Agent": LearningAgent(
            experience_path=experience_path
        ),'''

new = '''"Learning Agent": LearningAgent(
            experience_path=(
                experience_path
                if experience_path is not None
                else PROJECT_ROOT / "data" / "learning_agent_experience_integration.json"
            )
        ),'''

assert old in source, "Expected factory code not found."

source = source.replace(old, new)

# Remove diagnostic print statements executed during module import
tree = ast.parse(source)

tree.body = [
    node for node in tree.body
    if not (
        isinstance(node, ast.Expr)
        and isinstance(node.value, ast.Call)
        and isinstance(node.value.func, ast.Name)
        and node.value.func.id == "print"
    )
]

clean_source = ast.unparse(tree) + "\n"

compile(clean_source, str(module_path), "exec")

module_path.write_text(clean_source, encoding="utf-8")

print("MODULE CLEANUP: PASS")
print("Python syntax: PASS")
print("Integration learning database separated: PASS")
print("Original demonstration learning records preserved: PASS")

MODULE CLEANUP: PASS
Python syntax: PASS
Integration learning database separated: PASS
Original demonstration learning records preserved: PASS


In [41]:
# STEP 8.10: VERIFY CLEAN EXPORTED MODULE

import importlib
import json

import incd_assist.classical_agents as classical_module

# Reload the cleaned module
importlib.invalidate_caches()
classical_module = importlib.reload(classical_module)

# Use a separate integration learning database
integration_learning_file = (
    PROJECT_ROOT / "data" / "learning_agent_experience_integration.json"
)

# Create all five agents
final_agents = classical_module.create_classical_agents(
    experience_path=integration_learning_file
)

print("=" * 65)
print("FINAL MODULE VERIFICATION")
print("=" * 65)

for name, agent in final_agents.items():
    print("PASS:", name)

print("\nTesting all agents on scenarios A, B and C...")

final_results = {}

for scenario_id, scenario in scenarios.items():

    final_results[scenario_id] = {}

    for agent_name, agent in final_agents.items():

        result = agent.decide(scenario)

        assert isinstance(result, AgentResult)

        final_results[scenario_id][agent_name] = {
            "agent_name": result.agent_name,
            "incident_summary": result.incident_summary,
            "likely_cause": result.likely_cause,
            "recommended_action": result.recommended_action,
            "evidence": list(result.evidence),
            "verification_steps": list(result.verification_steps),
            "limitations": list(result.limitations),
            "reason": result.reason
        }

# Save final integration results
final_output = (
    PROJECT_ROOT / "outputs" /
    "chakritha_final_results_for_vishruth.json"
)

with open(final_output, "w", encoding="utf-8") as file:
    json.dump(final_results, file, indent=4)

total = sum(len(items) for items in final_results.values())

print("\nTotal agents:", len(final_agents))
print("Total scenarios:", len(final_results))
print("Total results:", total)

assert total == 15

print("\nPASS: All 15 results generated.")
print("PASS: Common AgentResult structure verified.")
print("\nFinal results saved:")
print(final_output)

Learning Agent initialized.
Experience file: C:\Users\CHAKRITHA\Documents\Python Files- BITS\BITS material\Agentic AI\Assignment1\data\learning_agent_experience_integration.json
Existing feedback categories: []
FINAL MODULE VERIFICATION
PASS: Simple Reflex Agent
PASS: Model-Based Reflex Agent
PASS: Goal-Based Agent
PASS: Utility-Based Agent
PASS: Learning Agent

Testing all agents on scenarios A, B and C...

Total agents: 5
Total scenarios: 3
Total results: 15

PASS: All 15 results generated.
PASS: Common AgentResult structure verified.

Final results saved:
C:\Users\CHAKRITHA\Documents\Python Files- BITS\BITS material\Agentic AI\Assignment1\outputs\chakritha_final_results_for_vishruth.json


In [43]:
# STEP 8.11: CREATE CHAKRITHA HANDOFF PACKAGE

import shutil
import zipfile
import json

HANDOFF_DIR = OUTPUT_DIR / "chakritha_handoff"

HANDOFF_DIR.mkdir(parents=True, exist_ok=True)

# Files required for team integration
files_to_copy = {
    "classical_agents.py":
        PROJECT_ROOT / "incd_assist" / "classical_agents.py",

    "final_results.json":
        OUTPUT_DIR / "chakritha_final_results_for_vishruth.json",

    "evaluation.csv":
        OUTPUT_DIR / "chakritha_classical_evaluation.csv",

    "learning_experience.json":
        PROJECT_ROOT / "data" / "learning_agent_experience.json",

    "working_notebook.ipynb":
        PROJECT_ROOT / "notebooks" / "chakritha_classical_agents.ipynb"
}

# Verify source files
for filename, source in files_to_copy.items():
    assert source.exists(), f"Missing file: {source}"

# Copy files
for filename, source in files_to_copy.items():
    shutil.copy2(source, HANDOFF_DIR / filename)

# Create integration instructions
readme = """
CHAKRITHA - CLASSICAL AGENT HANDOFF
===================================

Five implemented agents:
1. Simple Reflex Agent
2. Model-Based Reflex Agent
3. Goal-Based Agent
4. Utility-Based Agent
5. Learning Agent

RESULTS
-------
Three shared scenarios: A, B and C.
Total classical agent results: 15.
Expected-cause matches: 15/15.
Scenario C historical DB warning evidence: 5/5.

INTEGRATION
-----------

Import:

from incd_assist.scenarios import load_scenarios
from incd_assist.classical_agents import create_classical_agents

scenarios = load_scenarios()
agents = create_classical_agents()

Run:

for scenario_id, scenario in scenarios.items():
    for name, agent in agents.items():
        result = agent.decide(scenario)
        print(scenario_id, name, result.likely_cause)

All agents implement:
decide(scenario) -> AgentResult

LEARNING FEEDBACK
-----------------

The Learning Agent supports:

agent = agents["Learning Agent"]

agent.record_feedback(
    scenario=scenarios["A"],
    action_name="Investigate and release stuck database connections",
    successful=True,
    notes="Record only after an outcome is verified."
)

The integration database is separate from the simulated
learning demonstration database.

IMPORTANT
---------
Agents recommend actions but do not execute production changes.
Obtain operator approval before any production recovery action.

The shared project must include the original incd_assist package,
scenario data and Python dependencies.

This handoff covers Chakritha's five classical agents.
Tanweer provides the LLM agent; Vishruth provides the Streamlit UI.
"""

(HANDOFF_DIR / "README.txt").write_text(
    readme,
    encoding="utf-8"
)

# Create ZIP file
zip_path = OUTPUT_DIR / "chakritha_handoff_for_vishruth.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zip_file:

    for file in HANDOFF_DIR.iterdir():
        zip_file.write(file, arcname=file.name)

print("=" * 60)
print("CHAKRITHA HANDOFF PACKAGE CREATED")
print("=" * 60)

for file in HANDOFF_DIR.iterdir():
    print("PASS:", file.name)

print("\nZIP FILE:")
print(zip_path)

print("\nHandoff package ready")

CHAKRITHA HANDOFF PACKAGE CREATED
PASS: classical_agents.py
PASS: evaluation.csv
PASS: final_results.json
PASS: learning_experience.json
PASS: README.txt
PASS: working_notebook.ipynb

ZIP FILE:
C:\Users\CHAKRITHA\Documents\Python Files- BITS\BITS material\Agentic AI\Assignment1\outputs\chakritha_handoff_for_vishruth.zip

Handoff package ready


In [45]:
import zipfile

zip_path = OUTPUT_DIR / "chakritha_handoff.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    print("ZIP integrity:", z.testzip() or "PASS")
    print("\nFiles inside package:")

    for filename in z.namelist():
        print("-", filename)

print("\nHANDOFF PACKAGE VERIFIED!")

ZIP integrity: PASS

Files inside package:
- classical_agents.py
- evaluation.csv
- final_results.json
- learning_experience.json
- README.txt
- working_notebook.ipynb

HANDOFF PACKAGE VERIFIED!
